In [ ]:
gold_df = spark.read.table('workspace.ecommerce.silver_retail')
gold_df.printSchema()
display(gold_df.limit(10))

print(f"Total number of records in :{gold_df.count()}")

In [ ]:
# Gold layer: Transaction-level calculation + KPI aggregation
# Step 1: Calculate transaction-line amount (Quantity * Price).
# Step 2: Aggregate the entire dataset to calculate business KPIs.

from pyspark.sql.functions import *

gold_df = gold_df.withColumn(
    "total_amount",
    col("Quantity") * col("Price")
)

gold_kpi_df = gold_df.agg(
    sum("total_amount").alias("total_revenue"),
    sum("Quantity").alias("total_quantity_sold"),
    countDistinct("Invoice").alias("total_orders"),
    countDistinct("Customer_ID").alias("unique_customers")
)

display(gold_kpi_df)

In [ ]:
#Monthly analysis of the data
monthly_sales_df = gold_df.groupBy(
     year("InvoiceDate").alias("year"),
    month("InvoiceDate").alias("month")
   
).agg(
    sum("total_amount").alias("total_revenue"),
    sum("Quantity").alias("total_quantity_sold"),
    countDistinct("Invoice").alias("total_orders"),
    countDistinct("Customer_ID").alias("unique_customers")
)

display(monthly_sales_df)

In [ ]:
#Product performance analytics

product_performance_analytics_df = gold_df.groupBy("StockCode","Description")\
.agg(
    sum(col("Quantity")).alias("top_selling_products_by_quantity"),
    sum(col("total_amount")).alias("product_generating_highest_revenue"),
    countDistinct(col("Invoice")).alias("count_of_invoices_per_product"),
    avg(col("Price")).alias("average_price_per_product"))

display(product_performance_analytics_df.orderBy(col("top_selling_products_by_quantity").desc()))

In [ ]:
#Country Performance Analytics
from pyspark.sql.functions import *
country_performance_df = gold_df.groupBy(col("Country"))\
    .agg(
    sum(col("total_amount")).alias("total_revenue"),
    sum("Quantity").alias("total_quantity_sold"),
    countDistinct("Invoice").alias("total_orders"),
    countDistinct("Customer_ID").alias("unique_customers")
)
country_performance_df = country_performance_df.withColumn("average_order_value",col("total_revenue")/col("total_orders"))
display(country_performance_df.orderBy(col("total_revenue").desc()))


In [ ]:
# Customer Analytics

customer_df = gold_df.groupBy("Customer_ID").agg(
    sum(col("total_amount")).alias("total_revenue"),
    sum("Quantity").alias("total_quantity_sold"),
    countDistinct("Invoice").alias("total_orders")
)

customer_df = customer_df.withColumn(
    "average_order_value",
    col("total_revenue") / col("total_orders")
)

display(customer_df.orderBy(col("total_orders").desc()))

In [ ]:
#Ranking for the most spent customer to the least
from pyspark.sql.window import * 
window_spec = Window.partitionBy().orderBy(desc("total_revenue"))

customer_df = customer_df.withColumn("rank",dense_rank().over(window_spec))
display(customer_df)

In [ ]:
#Product aggregation by country
from pyspark.sql.window import *
from pyspark.sql.functions import *

Window_spec_country = Window.partitionBy("Country").orderBy(desc("total_revenue"))

product_aggregation_df = gold_df.groupBy("Country","StockCode","Description").agg(
    sum("total_amount").alias("total_revenue"),
    sum("Quantity").alias("total_quantity_sold")
)

product_aggregation_df = product_aggregation_df.withColumn("rank",dense_rank().over(Window_spec_country)).filter(col("rank") <= 5)
display(product_aggregation_df)

In [ ]:
# Month-over-Month Revenue Growth

mom_revenue_df = monthly_sales_df.select(
    col("year"),
    col("month"),
    col("total_revenue").alias("monthly_revenue")
)

window_spec = Window.partitionBy().orderBy("year", "month")

mom_revenue_df = (
    mom_revenue_df
    .withColumn(
        "previous_month_revenue",
        lag("monthly_revenue", 1).over(window_spec)
    )
    .withColumn(
        "revenue_difference",
        col("monthly_revenue") - col("previous_month_revenue")
    )
    .withColumn(
        "growth_percentage",
        (col("revenue_difference") / col("previous_month_revenue")) * 100
    )
)

display(mom_revenue_df)

In [ ]:
#Customer Retention & Repeat Purchase Analytics

customer_retention_df = customer_df.filter(col("Customer_ID").isNotNull())
customer_retention_df = customer_retention_df.withColumn("customer_type",when(col("total_orders") == 1,"One Time Buyer").otherwise("Repeat Buyer"))

customer_retention_summary_df = customer_retention_df.agg(
    countDistinct("Customer_ID").alias("total_customers"),
    sum(when(col("customer_type") == "One Time Buyer", 1).otherwise(0)).alias("one_time_customers"),
    sum(when(col("customer_type") == "Repeat Buyer", 1).otherwise(0)).alias("repeat_customers"),
    avg("total_orders").alias("average_orders_per_customer"),
    sum(when(col("customer_type") == "Repeat Buyer", col("total_revenue")).otherwise(0)).alias("repeat_customer_revenue")
)

customer_retention_summary_df = customer_retention_summary_df.withColumn(
    "repeat_purchase_rate",
    (col("repeat_customers") / col("total_customers")) * 100
)

display(customer_retention_summary_df)

In [ ]:
#Customer Segmentation
customer_segmentation_df = customer_df.filter(col("Customer_ID").isNotNull())

customer_segmentation_df = customer_segmentation_df.withColumn(
    "customer_segment",
    when(col("total_revenue") >= 10000, "High Value")
    .when(col("total_revenue") >= 1000, "Mid Value")
    .otherwise("Low Value")
)

customer_segment_summary_df = customer_segmentation_df.groupBy("customer_segment").agg(
    countDistinct("Customer_ID").alias("total_customers"),
    sum("total_revenue").alias("segment_revenue"),
    sum("total_orders").alias("total_orders"),
    avg("total_revenue").alias("average_revenue_per_customer"),
    avg("total_orders").alias("average_orders_per_customer")
).orderBy(desc("segment_revenue"))

display(customer_segment_summary_df)

In [ ]:
# Consolidated Business KPI Table

consolidated_kpi_df = gold_df.agg(
    sum("total_amount").alias("total_revenue"),
    sum("Quantity").alias("total_quantity_sold"),
    countDistinct("Invoice").alias("total_orders"),
    countDistinct("Customer_ID").alias("total_customers")
)

consolidated_kpi_df = consolidated_kpi_df.withColumn(
    "average_order_value",
    col("total_revenue") / col("total_orders")
)

customer_kpi_df = customer_df.filter(col("Customer_ID").isNotNull()).agg(
    avg("total_revenue").alias("average_revenue_per_customer"),
    avg("total_orders").alias("average_orders_per_customer")
)

customer_retention_kpi_df = customer_retention_summary_df.select(
    "one_time_customers",
    "repeat_customers",
    "repeat_purchase_rate",
    "repeat_customer_revenue"
)

consolidated_kpi_df = consolidated_kpi_df.crossJoin(customer_kpi_df).crossJoin(customer_retention_kpi_df)

display(consolidated_kpi_df)

In [ ]:
#Reconcile Silver vs Gold source
# Load Silver source for reconciliation
silver_validation_df = spark.read.table("workspace.ecommerce.silver_retail")

# Silver-level metrics
silver_metrics_df = silver_validation_df.agg(
    count("*").alias("silver_row_count"),
    sum(col("Quantity") * col("Price")).alias("silver_total_revenue"),
    sum("Quantity").alias("silver_total_quantity")
)

# Gold-level metrics
gold_metrics_df = gold_df.agg(
    count("*").alias("gold_row_count"),
    sum("total_amount").alias("gold_total_revenue"),
    sum("Quantity").alias("gold_total_quantity")
)

# Compare Silver and Gold metrics
reconciliation_df = silver_metrics_df.crossJoin(gold_metrics_df)

display(reconciliation_df)

In [ ]:
#Gold data quality validation
# Validate Gold transaction-level business rules

gold_quality_check_df = gold_df.agg(
    sum(when(col("Quantity") <= 0, 1).otherwise(0)).alias("invalid_quantity_records"),
    sum(when(col("Price") < 0, 1).otherwise(0)).alias("invalid_price_records"),
    sum(when(col("total_amount") < 0, 1).otherwise(0)).alias("invalid_total_amount_records"),
    sum(when(col("total_amount").isNull(), 1).otherwise(0)).alias("null_total_amount_records"),
    sum(when(col("Invoice").isNull(), 1).otherwise(0)).alias("null_invoice_records")
)

display(gold_quality_check_df)

In [ ]:
#Validate consolidated KPI results
# Independently calculate customer-level metrics for comparison
identified_customer_validation_df = customer_df.filter(
    col("Customer_ID").isNotNull()
).agg(
    sum("total_revenue").alias("identified_customer_revenue"),
    sum("total_orders").alias("identified_customer_orders"),
    countDistinct("Customer_ID").alias("identified_customers")
)

display(identified_customer_validation_df)
display(consolidated_kpi_df)

In [ ]:
# Persist Gold Analytics Tables as Delta

# 1. Monthly Sales Analytics
monthly_sales_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce.gold_monthly_sales")

# 2. Product Performance Analytics
product_performance_analytics_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce.gold_product_performance")

# 3. Country Performance Analytics
country_performance_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce.gold_country_performance")

# 4. Customer Analytics
customer_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce.gold_customer_analytics")

# 5. Top 5 Products per Country
product_aggregation_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce.gold_top5_products_country")

# 6. Month-over-Month Revenue Growth
mom_revenue_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce.gold_mom_revenue")

# 7. Customer Retention & Repeat Purchase Analytics
customer_retention_summary_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce.gold_customer_retention")

# 8. Customer Segmentation
customer_segment_summary_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce.gold_customer_segmentation")

# 9. Consolidated Business KPI Table
consolidated_kpi_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce.gold_business_kpi")

In [ ]:
# Verify Gold tables exist and are accessible

gold_tables = [
    "gold_monthly_sales",
    "gold_product_performance",
    "gold_country_performance",
    "gold_customer_analytics",
    "gold_top5_products_country",
    "gold_mom_revenue",
    "gold_customer_retention",
    "gold_customer_segmentation",
    "gold_business_kpi"
]

for table in gold_tables:
    table_name = f"workspace.ecommerce.{table}"

    print(f"\nVerifying table: {table_name}")

    df = spark.read.table(table_name)

    print(f"Row count: {df.count()}")
    df.printSchema()